# Procesamiento del Inventario de Puntos — Valle de Aburrá
### `MenM_VdeA.gpkg`

Este notebook documenta el flujo completo de procesamiento del inventario de puntos del Valle de Aburrá:

1. **Revisión** del archivo espacial original  
2. **Limpieza** del GeoPackage (ID único, duplicados, columnas, coordenada Z)  
3. **Extracción** de elevación y pendiente a partir del DEM  
4. **Filtrado** final a puntos con datos válidos  
5. **Visualización** — distribución espacial (scatter + histogramas marginales)  
6. **Visualización** — figura adicional  

---
**Estructura de carpetas:**
```
LGCP/
├── CODE/   ← este notebook
├── DATA/   ← archivos de entrada y salida (.gpkg, .tif)
└── FIG/    ← figuras generadas (.png)
```

**Archivos de entrada:**
- `DATA/MenM_VdeA.gpkg` — inventario de puntos original  
- `DATA/MDT_05001_20241102-002.tif` — DEM a 1 m (MAGNA-SIRGAS / Origen Nacional)

**Archivos de salida:**
- `DATA/MenM_VdeA_clean.gpkg` — puntos limpios (paso 2)
- `DATA/MenM_VdeA_dem.gpkg` — inventario final con `año`, `elevacion_m`, `pendiente_deg`
- `FIG/MenM_VdeA_spatial_distribution.png` — figura distribución espacial

## 0. Instalación de dependencias y rutas

In [ ]:
# Instalar dependencias si no están disponibles
# !pip install geopandas fiona rasterio pyproj contextily matplotlib psutil

import warnings, os
warnings.filterwarnings('ignore')

# ── Rutas (el notebook vive en CODE/, los datos en DATA/, las figuras en FIG/) ──
BASE_DIR   = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))

DATA_DIR   = os.path.join(BASE_DIR, "DATA") + os.sep
FIG_DIR    = os.path.join(BASE_DIR, "FIG")  + os.sep

# Crear FIG si no existe
os.makedirs(FIG_DIR, exist_ok=True)

PATH_PTS   = DATA_DIR + "MenM_VdeA.gpkg"                 # puntos originales
PATH_DEM   = DATA_DIR + "MDT_05001_20241102-002.tif"      # DEM 1 m
PATH_CLEAN = DATA_DIR + "MenM_VdeA_clean.gpkg"           # salida limpieza
PATH_OUT   = DATA_DIR + "MenM_VdeA_dem.gpkg"             # salida final

PATH_FIG1  = FIG_DIR  + "MenM_VdeA_spatial_distribution.png"

print("Rutas configuradas:")
print(f"  DATA → {DATA_DIR}")
print(f"  FIG  → {FIG_DIR}")

---
## 1. Revisión del archivo original

In [ ]:
import geopandas as gpd
import fiona
import pandas as pd
import numpy as np

# ── Capas disponibles en el GeoPackage ───────────────────────────────────────
layers = fiona.listlayers(PATH_PTS)
print("Capas disponibles:", layers)

# ── Carga y metadatos generales ───────────────────────────────────────────────
gdf = gpd.read_file(PATH_PTS)

print(f"\n{'='*45}")
print("  METADATOS GENERALES")
print(f"{'='*45}")
print(f"  Capa:            {layers[0]}")
print(f"  Tipo geometría:  {gdf.geometry.geom_type.unique()}")
print(f"  Nº registros:    {len(gdf)}")
print(f"  Nº columnas:     {len(gdf.columns)}")
print(f"  CRS:             {gdf.crs}")
print(f"  Bounding Box:    {gdf.total_bounds.round(5)}")

print(f"\n  Columnas y tipos:\n{gdf.dtypes.to_string()}")

print(f"\n  Valores nulos por columna:")
nulls = gdf.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "  Sin valores nulos")

In [ ]:
# ── Análisis de duplicados y coordenada Z ────────────────────────────────────
dup_geom = gdf.geometry.duplicated().sum()
dup_id   = gdf['id'].duplicated().sum()
has_z    = gdf.geometry.has_z.all()
z_vals   = gdf.geometry.apply(lambda g: g.z if hasattr(g, 'z') else np.nan)

print(f"Geometrías duplicadas: {dup_geom}")
print(f"IDs duplicados:        {dup_id}")
print(f"Tiene coordenada Z:    {has_z}  (Z únicas: {z_vals.nunique()}, min={z_vals.min()}, max={z_vals.max()})")

# ── Distribución temporal (campo 'Name') ─────────────────────────────────────
print(f"\nDistribución del campo 'Name':")
print(gdf['Name'].value_counts().sort_index().to_string())

# ── Distribución por capa de origen ──────────────────────────────────────────
print(f"\nRegistros por capa de origen:")
print(gdf['layer'].value_counts().to_string())

---
## 2. Limpieza del GeoPackage

Operaciones aplicadas:
- Eliminar **geometrías duplicadas** (3 registros)
- Eliminar **columnas 100% vacías**: `timestamp`, `begin`, `end`, `altitudeMode`, `drawOrder`, `icon`
- Conservar solo el campo `Name` → renombrar a **`año`**
- Eliminar **coordenada Z** (siempre = 0, no informativa)
- Generar **`ID_punto`** único correlativo

> Los 204 registros con `año` no estándar (`Placemark`, `20112`, nombres de lugar, etc.) se conservan para corrección manual.

In [ ]:
import os
from shapely.geometry import Point
from fiona.crs import from_epsg

gdf = gpd.read_file(PATH_PTS)
n_orig = len(gdf)

# 1. Eliminar geometrías duplicadas (conservar primera ocurrencia)
wkt      = gdf.geometry.apply(lambda g: g.wkt)
dup_mask = wkt.duplicated(keep='first')
gdf      = gdf[~dup_mask].reset_index(drop=True)
print(f"Geometrías duplicadas eliminadas: {dup_mask.sum()}  → quedan {len(gdf)}")

# 2. Eliminar columnas 100% vacías
empty_cols = [c for c in gdf.columns if c != 'geometry' and gdf[c].isnull().all()]
gdf = gdf.drop(columns=empty_cols)
print(f"Columnas vacías eliminadas: {empty_cols}")

# 3. Conservar solo Name + geometry → renombrar a 'año'
gdf = gdf[['Name', 'geometry']].rename(columns={'Name': 'año'})

# 4. Eliminar coordenada Z (Point Z → Point 2D)
gdf['geometry'] = gdf.geometry.apply(lambda g: Point(g.x, g.y) if g.has_z else g)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')
print(f"¿Quedan coordenadas Z? {gdf.geometry.has_z.any()}")

# 5. Generar ID único correlativo
# Nota: 'fid' es nombre reservado en GeoPackage → se usa 'ID_punto'
gdf.insert(0, 'ID_punto', range(1, len(gdf) + 1))
print(f"ID único generado: ID_punto 1 … {len(gdf)}")

# 6. Guardar
schema = {
    'geometry': 'Point',
    'properties': {'ID_punto': 'int', 'año': 'str'}
}
if os.path.exists(PATH_CLEAN):
    os.remove(PATH_CLEAN)

with fiona.open(PATH_CLEAN, 'w', driver='GPKG', schema=schema,
                crs=from_epsg(4326), layer='MenM_VdeA') as dst:
    for _, row in gdf.iterrows():
        dst.write({
            'geometry': {'type': 'Point', 'coordinates': (row.geometry.x, row.geometry.y)},
            'properties': {
                'ID_punto': int(row['ID_punto']),
                'año':      str(row['año']) if pd.notna(row['año']) else None
            }
        })

print(f"\n✓ Guardado: {PATH_CLEAN}")
print(f"  Registros: {len(gdf)}  |  Columnas: {list(gdf.columns)}")

# ── Resumen de valores no estándar en 'año' ───────────────────────────────────
import re
invalid_años = gdf[~gdf['año'].str.match(r'^\d{4}$', na=False)]['año']
print(f"\nRegistros con 'año' no estándar: {len(invalid_años)}")
print(invalid_años.value_counts().to_string())

---
## 3. Extracción de atributos topográficos desde el DEM

**DEM:** `MDT_05001_20241102-002.tif` — resolución 1 m, CRS MAGNA-SIRGAS / Origen Nacional (EPSG:9377)

**Atributos calculados:**

| Atributo | Resolución DEM | Método |
|---|---|---|
| `elevacion_m` | 1 m (franjas) | Lookup directo de píxel |
| `pendiente_deg` | 1 m (franjas) | Horn kernel 3×3 |
| `aspecto_deg` | 1 m (franjas) | Horn kernel 3×3 → azimut geográfico |
| `area_acum_m2` | 10 m (remuestreo) | pysheds D8 flow accumulation |

**Estrategia de procesamiento por memoria:**
- Elevación, pendiente y aspecto: franjas de 3.000 filas con 1 fila de buffer (DEM 1m, ~3.1 GB)
- Área acumulada: DEM remuestreado a 10m (~26 MB en RAM) → pysheds D8 → área en m²

**Convenciones:**
- Aspecto: 0° = Norte, 90° = Este, 180° = Sur, 270° = Oeste  
- Área acumulada: número de celdas 10m×10m que drenan al punto × 100 m²/celda

In [ ]:
import rasterio
from rasterio.enums import Resampling
from pysheds.grid import Grid

DEM_10M = DATA_DIR + "dem_10m.tif"   # DEM remuestreado (generado aquí)

# ── Inspección del DEM 1m ─────────────────────────────────────────────────────
with rasterio.open(PATH_DEM) as src:
    print(f"DEM 1m:  {src.width}×{src.height} px  |  res={src.res}  |  nodata={src.nodata}")
    size_gb = src.width * src.height * 4 / 1024**3
    print(f"  Tamaño RAM completo: {size_gb:.2f} GB  → procesamiento por franjas")

# ── Generar DEM 10m para flow accumulation ───────────────────────────────────
FACTOR = 10
with rasterio.open(PATH_DEM) as src:
    new_h = src.height // FACTOR
    new_w = src.width  // FACTOR
    data  = src.read(1, out_shape=(new_h, new_w),
                     resampling=Resampling.bilinear)
    new_tf  = src.transform * src.transform.scale(src.width/new_w, src.height/new_h)
    profile = src.profile.copy()
    profile.update(width=new_w, height=new_h, transform=new_tf,
                   dtype='float32', compress='lzw', nodata=src.nodata, count=1)

with rasterio.open(DEM_10M, 'w', **profile) as dst:
    dst.write(data.astype('float32'), 1)

print(f"\nDEM 10m: {new_w}×{new_h} px  |  RAM: {new_w*new_h*4/1024**2:.1f} MB")
print("✓ DEM 10m generado")

In [ ]:
# ── Funciones de análisis topográfico (kernel Horn 3×3) ──────────────────────

def horn_slope(data, nodata, res):
    """Pendiente en grados. Retorna shape (rows-2, cols-2)."""
    d = data.astype(np.float32)
    d[d == nodata] = np.nan
    dzdx = ((d[:-2, 2:] + 2*d[1:-1, 2:] + d[2:, 2:]) -
            (d[:-2,:-2] + 2*d[1:-1,:-2] + d[2:,:-2])) / (8.0 * res)
    dzdy = ((d[2:,:-2] + 2*d[2:,1:-1] + d[2:, 2:]) -
            (d[:-2,:-2] + 2*d[:-2,1:-1] + d[:-2, 2:])) / (8.0 * res)
    return np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))

def horn_aspect(data, nodata):
    """
    Aspecto en grados (azimut geográfico).
    0° = Norte, 90° = Este, 180° = Sur, 270° = Oeste.
    Retorna shape (rows-2, cols-2).
    """
    d = data.astype(np.float32)
    d[d == nodata] = np.nan
    dzdx = ((d[:-2, 2:] + 2*d[1:-1, 2:] + d[2:, 2:]) -
            (d[:-2,:-2] + 2*d[1:-1,:-2] + d[2:,:-2])) / 8.0
    dzdy = ((d[2:,:-2] + 2*d[2:,1:-1] + d[2:, 2:]) -
            (d[:-2,:-2] + 2*d[:-2,1:-1] + d[:-2, 2:])) / 8.0
    asp  = np.degrees(np.arctan2(-dzdy, dzdx))
    geo  = 90.0 - asp
    return np.where(geo < 0, geo + 360.0, geo).astype(np.float32)


# ── Leer puntos limpios y reproyectar ─────────────────────────────────────────
with fiona.open(PATH_CLEAN, 'r') as src:
    records = [{'ID_punto': f['properties']['ID_punto'],
                'año':      f['properties']['año'],
                'geometry': Point(f['geometry']['coordinates'])}
               for f in src]
    pts_crs = src.crs

gdf_pts = gpd.GeoDataFrame(records, geometry='geometry', crs=pts_crs)
print(f"Puntos cargados: {len(gdf_pts)}")

# ── Metadata DEM 1m ───────────────────────────────────────────────────────────
with rasterio.open(PATH_DEM) as src:
    dem_crs    = src.crs
    dem_bounds = src.bounds
    dem_tf     = src.transform
    dem_res    = src.res[0]
    dem_nodata = src.nodata
    dem_w      = src.width
    dem_h      = src.height

gdf_proj = gdf_pts.to_crs(dem_crs)

minx, miny, maxx, maxy = dem_bounds
mask = ((gdf_proj.geometry.x > minx) & (gdf_proj.geometry.x < maxx) &
        (gdf_proj.geometry.y > miny) & (gdf_proj.geometry.y < maxy))
gdf_clip = gdf_proj[mask].copy().reset_index(drop=True)
print(f"Puntos dentro del DEM: {len(gdf_clip)} / {len(gdf_proj)}")

xs = gdf_clip.geometry.x.values
ys = gdf_clip.geometry.y.values
gdf_clip['_col'] = np.floor((xs - dem_tf.c) / dem_tf.a).astype(int)
gdf_clip['_row'] = np.floor((ys - dem_tf.f) / dem_tf.e).astype(int)

In [ ]:
# ── Procesamiento por franjas: elevación, pendiente y aspecto (DEM 1m) ────────
STRIP = 3000

elev_vals   = np.full(len(gdf_clip), np.nan, dtype=np.float32)
slope_vals  = np.full(len(gdf_clip), np.nan, dtype=np.float32)
aspect_vals = np.full(len(gdf_clip), np.nan, dtype=np.float32)

strip_starts = list(range(0, dem_h, STRIP))
print(f"Franjas a procesar: {len(strip_starts)}")

with rasterio.open(PATH_DEM) as src:
    for i_strip, row_start in enumerate(strip_starts):

        r0 = max(0, row_start - 1)        # buffer superior
        r1 = min(dem_h, row_start + STRIP + 1)  # buffer inferior

        pts_in = gdf_clip[
            (gdf_clip['_row'] >= row_start) &
            (gdf_clip['_row'] <  row_start + STRIP)
        ].index
        if len(pts_in) == 0:
            continue

        window     = rasterio.windows.Window(0, r0, dem_w, r1 - r0)
        strip_data = src.read(1, window=window)

        local_rows = gdf_clip.loc[pts_in, '_row'].values - r0
        local_cols = gdf_clip.loc[pts_in, '_col'].values
        valid = ((local_rows >= 0) & (local_rows < strip_data.shape[0]) &
                 (local_cols >= 0) & (local_cols < strip_data.shape[1]))

        # ── Elevación (lookup directo) ────────────────────────────────────────
        vals = strip_data[local_rows[valid], local_cols[valid]].astype(np.float32)
        vals[vals == dem_nodata] = np.nan
        elev_vals[pts_in[valid]] = vals

        # ── Pendiente y aspecto (kernel Horn 3×3) ─────────────────────────────
        if strip_data.shape[0] >= 3:
            slp = horn_slope(strip_data, dem_nodata, dem_res)
            asp = horn_aspect(strip_data, dem_nodata)

            slp_rows = local_rows[valid] - 1
            slp_cols = local_cols[valid] - 1
            in_k = ((slp_rows >= 0) & (slp_rows < slp.shape[0]) &
                    (slp_cols >= 0) & (slp_cols < slp.shape[1]))

            slope_vals [pts_in[valid][in_k]] = slp[slp_rows[in_k], slp_cols[in_k]]
            aspect_vals[pts_in[valid][in_k]] = asp[slp_rows[in_k], slp_cols[in_k]]

        pct = (i_strip + 1) / len(strip_starts) * 100
        print(f"  Franja {i_strip+1}/{len(strip_starts)} ({pct:.0f}%) "
              f"— filas {row_start}–{row_start+STRIP} — {len(pts_in)} pts", flush=True)

gdf_clip['elevacion_m']   = np.round(elev_vals,   1)
gdf_clip['pendiente_deg'] = np.round(slope_vals,  2)
gdf_clip['aspecto_deg']   = np.round(aspect_vals, 1)
gdf_out = gdf_clip.to_crs('EPSG:4326')

print(f"\nElevación  — min: {np.nanmin(elev_vals):.0f} m   max: {np.nanmax(elev_vals):.0f} m")
print(f"Pendiente  — min: {np.nanmin(slope_vals):.1f}°  max: {np.nanmax(slope_vals):.1f}°")
print(f"Aspecto    — min: {np.nanmin(aspect_vals):.1f}°  max: {np.nanmax(aspect_vals):.1f}°")


# ── Flow accumulation con pysheds sobre DEM 10m ───────────────────────────────
print("\nCalculando área acumulada (D8 flow accumulation)…")

grid        = Grid.from_raster(DEM_10M)
dem_ps      = grid.read_raster(DEM_10M)
pit_filled  = grid.fill_pits(dem_ps)
flooded     = grid.fill_depressions(pit_filled)
inflated    = grid.resolve_flats(flooded)
fdir        = grid.flowdir(inflated)
acc         = grid.accumulation(fdir)          # nº de celdas que drenan al píxel
acc_m2_arr  = np.array(acc, dtype=np.float64) * 100.0   # × 100 m²/celda (10m×10m)

print(f"Área acumulada máx: {acc_m2_arr.max()/1e6:.2f} km²")

# Metadata DEM 10m
with rasterio.open(DEM_10M) as src:
    tf_10m = src.transform
    h_10m  = src.height
    w_10m  = src.width

# Índices de los puntos en el raster 10m
xs_out = gdf_out.to_crs(grid.crs).geometry.x.values
ys_out = gdf_out.to_crs(grid.crs).geometry.y.values
col_10 = np.floor((xs_out - tf_10m.c) / tf_10m.a).astype(int)
row_10 = np.floor((ys_out - tf_10m.f) / tf_10m.e).astype(int)

acc_vals = np.full(len(gdf_out), np.nan, dtype=np.float64)
ok = ((row_10 >= 0) & (row_10 < h_10m) & (col_10 >= 0) & (col_10 < w_10m))
acc_vals[ok] = acc_m2_arr[row_10[ok], col_10[ok]]

gdf_out['area_acum_m2'] = np.round(acc_vals, 1)
print(f"Área acumulada — min: {np.nanmin(acc_vals):.0f} m²  "
      f"max: {np.nanmax(acc_vals):.0f} m²  "
      f"media: {np.nanmean(acc_vals):.0f} m²")

---
## 4. Filtrado final y guardado

Se conservan únicamente los puntos con valores válidos en **los 4 atributos topográficos** (elevación, pendiente, aspecto y área acumulada).

In [ ]:
# ── Filtrar a puntos con todos los atributos válidos ─────────────────────────
valid_mask = (gdf_out['elevacion_m'].notna()   &
              gdf_out['pendiente_deg'].notna()  &
              gdf_out['aspecto_deg'].notna()    &
              gdf_out['area_acum_m2'].notna())

gdf_final = gdf_out[valid_mask].copy().reset_index(drop=True)
print(f"Registros con todos los atributos válidos: {len(gdf_final)}")
print(f"Registros descartados (algún NoData):      {(~valid_mask).sum()}")

# ── Guardar GeoPackage final ──────────────────────────────────────────────────
schema = {
    'geometry': 'Point',
    'properties': {
        'ID_punto':      'int',
        'año':           'str',
        'elevacion_m':   'float',
        'pendiente_deg': 'float',
        'aspecto_deg':   'float',
        'area_acum_m2':  'float'
    }
}
if os.path.exists(PATH_OUT):
    os.remove(PATH_OUT)

with fiona.open(PATH_OUT, 'w', driver='GPKG', schema=schema,
                crs=from_epsg(4326), layer='MenM_VdeA') as dst:
    for _, row in gdf_final.iterrows():
        def fv(v): return float(v) if pd.notna(v) else None
        dst.write({
            'geometry': {'type': 'Point',
                         'coordinates': (row.geometry.x, row.geometry.y)},
            'properties': {
                'ID_punto':      int(row['ID_punto']),
                'año':           str(row['año']) if pd.notna(row['año']) else None,
                'elevacion_m':   fv(row['elevacion_m']),
                'pendiente_deg': fv(row['pendiente_deg']),
                'aspecto_deg':   fv(row['aspecto_deg']),
                'area_acum_m2':  fv(row['area_acum_m2'])
            }
        })

print(f"\n✓ Guardado: {PATH_OUT}")
print(f"  Columnas: ID_punto | año | elevacion_m | pendiente_deg | aspecto_deg | area_acum_m2 | geometry")
print(f"  CRS:      EPSG:4326 (WGS84)")

# ── Resumen estadístico ───────────────────────────────────────────────────────
print(f"\n{'Atributo':<20} {'Min':>10} {'Max':>10} {'Media':>10} {'Mediana':>10}")
print("-" * 55)
for col, unit in [('elevacion_m','m'), ('pendiente_deg','°'),
                  ('aspecto_deg','°'), ('area_acum_m2','m²')]:
    s = gdf_final[col]
    print(f"  {col:<18} {s.min():>9.1f} {s.max():>9.1f} {s.mean():>9.1f} {s.median():>9.1f}  {unit}")

---
## 5. Visualización: distribución espacial (scatter + histogramas marginales)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import AutoMinorLocator

lons = gdf_final.geometry.x.values
lats = gdf_final.geometry.y.values

def make_joint_plot(lons, lats, out_path, dpi=300):
    """
    Genera un scatter plot con histogramas marginales
    (distribución espacial de puntos) y lo guarda en PNG.
    """
    fig = plt.figure(figsize=(8, 8), dpi=dpi)
    fig.patch.set_facecolor('white')

    gs = gridspec.GridSpec(
        2, 2,
        width_ratios=[4, 1],
        height_ratios=[1, 4],
        hspace=0.04,
        wspace=0.04
    )

    ax_hist_top   = fig.add_subplot(gs[0, 0])   # histograma superior (longitud)
    ax_scatter    = fig.add_subplot(gs[1, 0])   # scatter central
    ax_hist_right = fig.add_subplot(gs[1, 1])   # histograma derecho (latitud)
    ax_corner     = fig.add_subplot(gs[0, 1])   # esquina vacía
    ax_corner.set_visible(False)

    BLUE  = '#3a7abf'
    ALPHA = 0.35
    BINS  = 40

    # ── Scatter central ───────────────────────────────────────────────────────
    ax_scatter.scatter(lons, lats, s=3, color=BLUE, alpha=ALPHA,
                       linewidths=0, rasterized=True)
    ax_scatter.set_xlabel('Longitud', fontsize=11, labelpad=6)
    ax_scatter.set_ylabel('Latitud',  fontsize=11, labelpad=6)
    ax_scatter.grid(True, alpha=0.25, linewidth=0.5, linestyle='--')
    ax_scatter.tick_params(axis='both', labelsize=9)
    ax_scatter.xaxis.set_minor_locator(AutoMinorLocator())
    ax_scatter.yaxis.set_minor_locator(AutoMinorLocator())
    ax_scatter.set_xlim(lons.min() - 0.005, lons.max() + 0.005)
    ax_scatter.set_ylim(lats.min() - 0.005, lats.max() + 0.005)
    ax_scatter.annotate(
        f'n = {len(lons):,}',
        xy=(0.97, 0.03), xycoords='axes fraction',
        ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#aaaaaa', alpha=0.8)
    )

    # ── Histograma superior (longitud) ────────────────────────────────────────
    ax_hist_top.hist(lons, bins=BINS, color=BLUE, alpha=0.75, edgecolor='none')
    ax_hist_top.set_xlim(ax_scatter.get_xlim())
    ax_hist_top.set_ylabel('Frecuencia', fontsize=8)
    ax_hist_top.grid(True, axis='y', alpha=0.25, linewidth=0.5, linestyle='--')
    ax_hist_top.tick_params(labelbottom=False, labelsize=8)
    ax_hist_top.spines[['top', 'right']].set_visible(False)

    # ── Histograma derecho (latitud) ──────────────────────────────────────────
    ax_hist_right.hist(lats, bins=BINS, color=BLUE, alpha=0.75,
                       edgecolor='none', orientation='horizontal')
    ax_hist_right.set_ylim(ax_scatter.get_ylim())
    ax_hist_right.set_xlabel('Frecuencia', fontsize=8)
    ax_hist_right.grid(True, axis='x', alpha=0.25, linewidth=0.5, linestyle='--')
    ax_hist_right.tick_params(labelleft=False, labelsize=8)
    ax_hist_right.spines[['top', 'right']].set_visible(False)

    fig.suptitle('Distribución espacial del inventario de puntos\nValle de Aburrá',
                 fontsize=12, fontweight='bold', y=0.99)

    fig.savefig(out_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"✓ Figura guardada: {out_path}")
    return fig


fig = make_joint_plot(lons, lats, PATH_FIG1)
plt.show()

---
## 6. Visualización: histograma hexagonal de densidad de movimientos en masa

Mapa de densidad espacial usando `hexbin` con:
- **Mapa base** CartoDB Positron (requiere conexión a internet)
- **Polígonos** del inventario como contexto geoespacial
- **Hexbins ~250 m × 250 m** coloreados por conteo de eventos (`viridis`)
- **Colorbar** lateral con número de movimientos en masa por hexbin

> **Nota sobre resolución de bins:**  
> El inventario tiene 2.504 puntos distribuidos sobre ~24 × 22 km.  
> Con bins de 10 m el conteo máximo sería 2 (no produce densidad útil).  
> Con bins de 250 m el conteo máximo es 23, revelando claramente los focos de mayor concentración.  
> Ajusta `BIN_M` según la escala de análisis deseada.

In [ ]:
import contextily as ctx
import pyproj

# ── Rutas ─────────────────────────────────────────────────────────────────────
PATH_POLY  = DATA_DIR + "MenM_VdeA_Polygon.gpkg"
PATH_FIG2  = FIG_DIR  + "MenM_VdeA_hexbin_densidad.png"

# ── Parámetro principal ───────────────────────────────────────────────────────
BIN_M = 250   # tamaño de bin en metros (ajustar según escala de análisis)


def make_hexbin_density(path_pts, path_poly, out_path, bin_m=250, dpi=300):
    """
    Histograma hexagonal de densidad de movimientos en masa.

    El hexbin se calcula en coordenadas proyectadas MAGNA-SIRGAS (EPSG:9377)
    para garantizar bins de tamaño métrico exacto. Los hexágonos se dibujan
    sobre Web Mercator (EPSG:3857) para compatibilidad con el mapa base.

    Parámetros
    ----------
    path_pts  : str  — GeoPackage de puntos finales
    path_poly : str  — GeoPackage de polígonos de contexto
    out_path  : str  — ruta de salida PNG
    bin_m     : int  — lado del hexbin en metros (default 250)
    dpi       : int  — resolución de exportación
    """
    # ── Leer y reproyectar puntos ─────────────────────────────────────────────
    with fiona.open(path_pts, 'r') as src:
        records = [{'geometry': Point(f['geometry']['coordinates'])} for f in src]

    gdf      = gpd.GeoDataFrame(records, geometry='geometry', crs='EPSG:4326')
    gdf_9377 = gdf.to_crs(epsg=9377)

    xs_m = gdf_9377.geometry.x.values
    ys_m = gdf_9377.geometry.y.values

    # Reproyectar a Web Mercator para superponer sobre basemap
    tr_to_wm = pyproj.Transformer.from_crs('EPSG:9377', 'EPSG:3857', always_xy=True)
    xs_wm, ys_wm = tr_to_wm.transform(xs_m, ys_m)

    # ── Polígonos ─────────────────────────────────────────────────────────────
    gdf_poly_wm = gpd.read_file(path_poly).to_crs(epsg=3857)

    # ── gridsize: número de hexbins en X para alcanzar bin_m metros ──────────
    x_range  = xs_m.max() - xs_m.min()
    gridsize = int(x_range / bin_m)
    real_bin = x_range / gridsize
    print(f"gridsize = {gridsize}  |  bin real = {real_bin:.0f} m")

    # ── Extensión del mapa ────────────────────────────────────────────────────
    PAD  = 1800   # metros de margen en Web Mercator
    xmin, xmax = xs_wm.min() - PAD, xs_wm.max() + PAD
    ymin, ymax = ys_wm.min() - PAD, ys_wm.max() + PAD

    fig, ax = plt.subplots(figsize=(10, 10), dpi=dpi)
    fig.patch.set_facecolor('white')
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect('equal')

    # ── 1. Mapa base CartoDB Positron ─────────────────────────────────────────
    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom=12, zorder=1)
    except Exception:
        ax.set_facecolor('#f5f5f5')   # fallback sin conexión

    # ── 2. Polígonos de contexto ──────────────────────────────────────────────
    gdf_poly_wm.plot(ax=ax, facecolor='none', edgecolor='#90a4ae',
                     linewidth=0.4, alpha=0.4, zorder=2)

    # ── 3. Hexbin — conteo de movimientos en masa por celda ───────────────────
    hb = ax.hexbin(
        xs_wm, ys_wm,
        gridsize=gridsize,
        cmap='viridis',
        mincnt=1,             # ocultar celdas vacías
        linewidths=0.15,
        edgecolors='white',
        alpha=0.88,
        zorder=3
    )

    # ── 4. Colorbar ───────────────────────────────────────────────────────────
    cbar = fig.colorbar(hb, ax=ax, fraction=0.028, pad=0.015, aspect=32)
    cbar.set_label(f'Movimientos en masa por hexbin\n(bin ≈ {bin_m} m)',
                   fontsize=10, labelpad=10)
    cbar.ax.tick_params(labelsize=9)

    # ── 5. Estadísticas en consola ────────────────────────────────────────────
    counts = hb.get_array()
    active = counts[counts > 0]
    print(f"Hexbins con datos: {len(active)}")
    print(f"Conteo  max={int(active.max())}  |  mediana={np.median(active):.1f}  |  media={active.mean():.1f}")

    # ── 6. Ejes con coordenadas geográficas ───────────────────────────────────
    tr_wgs = pyproj.Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)
    xt = np.linspace(xmin, xmax, 5)
    yt = np.linspace(ymin, ymax, 5)
    ax.set_xticks(xt)
    ax.set_yticks(yt)
    ax.set_xticklabels([f"{tr_wgs.transform(x, ys_wm.mean())[0]:.3f}°" for x in xt],
                       fontsize=8, rotation=30, ha='right')
    ax.set_yticklabels([f"{tr_wgs.transform(xs_wm.mean(), y)[1]:.3f}°" for y in yt],
                       fontsize=8)
    ax.set_xlabel('Longitud', fontsize=10, labelpad=6)
    ax.set_ylabel('Latitud',  fontsize=10, labelpad=6)
    ax.grid(True, linewidth=0.3, alpha=0.3, linestyle='--', color='gray', zorder=0)

    ax.set_title(
        f'Densidad de Movimientos en Masa — Valle de Aburrá\n'
        f'Histograma hexagonal (bin ≈ {bin_m} m × {bin_m} m)',
        fontsize=12, fontweight='bold', pad=12
    )
    ax.text(0.01, 0.005, '© OpenStreetMap contributors © CARTO',
            transform=ax.transAxes, fontsize=6, color='#666')
    ax.annotate(f'n = {len(gdf):,}', xy=(0.97, 0.015), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#aaa', alpha=0.8))

    plt.tight_layout()
    fig.savefig(out_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"✓ Figura guardada: {out_path}")
    return fig


fig = make_hexbin_density(PATH_OUT, PATH_POLY, PATH_FIG2, bin_m=BIN_M)
plt.show()